# RDF / OWL + LangGraph — Learning Notebook

> **Use case:** Same warranty-diagnosis domain as the repo.
> Works **without** an API key (mock mode). Upgrades to real LLM + Neo4j in Part 6.

---

## Table of Contents
| | Section | What you build |
|--|--|--|
| **Part 1** | TBox — the schema | `owl:Class`, `owl:ObjectProperty`, `owl:DatatypeProperty`, OWL axioms |
| **Part 2** | ABox — the data | Instances, reified edges with confidence |
| **Part 3** | SPARQL | Query the graph — symptoms, failure modes, diagnostic paths |
| **Part 4** | LangGraph Agent | `@tool` → `StateGraph` → conditional edge loop |
| **Part 5** | Memory | `MemorySaver` + `thread_id` for multi-turn conversations |
| **Part 6** | Neo4j Upgrade | Drop-in swap: rdflib → `langchain_neo4j` |

In [27]:
# Install the two packages not yet in this venv
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "rdflib", "langchain-openai", "langchain-neo4j"])
print("Dependencies installed ✓")

Dependencies installed ✓


In [28]:
# ── Core rdflib imports ───────────────────────────────────────────────────────
import os

from rdflib import OWL, RDF, RDFS, XSD, BNode, Graph, Literal, Namespace, URIRef

# Our project namespace — same URI as the repo's warranty-diagnosis ontology
WD = Namespace("https://example.org/warranty-diagnosis#")
PROV = Namespace("http://www.w3.org/ns/prov#")


def new_graph():
    """Helper: create a Graph with standard prefixes already bound."""
    g = Graph()
    g.bind("wd", WD)
    g.bind("prov", PROV)
    return g


print("rdflib imported ✓")

rdflib imported ✓


---
## Part 1 — TBox: Defining the Schema

### The one rule that explains everything
Every single statement in RDF/OWL is a **triple**:
```
Subject  →  Predicate  →  Object
```

### Turtle punctuation cheat sheet
| Symbol | Means |
|--------|-------|
| `.` | End of all triples for this subject |
| `;` | Same subject, **new predicate** |
| `,` | Same subject **and** same predicate, new object |

### `a` is just a shortcut
`wd:Product a owl:Class` = `wd:Product rdf:type owl:Class`

### TBox vs ABox
| Term | Meaning | Analogy |
|------|---------|--------|
| **TBox** | Class + property definitions | TypeScript `interface` / DB schema |
| **ABox** | Actual instances | `const p: Product = {...}` / DB rows |

### The vocabulary ladder (what each keyword does)
```turtle
rdf:type            ← "is of type" (what something IS)
rdfs:label          ← human-readable name
rdfs:comment        ← description
rdfs:subClassOf     ← inheritance (A is a kind of B)
rdfs:domain         ← the "from" class of a property
rdfs:range          ← the "to" class / datatype of a property

owl:Class           ← declares a class  (stronger than rdfs:Class)
owl:ObjectProperty  ← relationship between two nodes (URI → URI)
owl:DatatypeProperty ← attribute with a literal value (URI → string/number)
owl:Ontology        ← the namespace/file declaration itself
```

In [29]:
# ── TBox Part A: Classes ───────────────────────────────────────────────────────
#
# Pattern for every class (three triples per class):
#   g.add( (WD.ClassName, RDF.type,     OWL.Class)           )  ← IS an OWL Class
#   g.add( (WD.ClassName, RDFS.label,   Literal('name'))     )  ← human-readable label
#   g.add( (WD.ClassName, RDFS.comment, Literal('desc'))     )  ← description
#
# In Turtle syntax this looks like:
#   wd:Product a owl:Class ;
#     rdfs:label   "Product" ;
#     rdfs:comment "Manufacturable product family." .

g_tbox = new_graph()

# Declare the Ontology header (the namespace itself)
ONTO = URIRef("https://example.org/warranty-diagnosis/ontology")
g_tbox.add((ONTO, RDF.type, OWL.Ontology))
g_tbox.add((ONTO, RDFS.label, Literal("Warranty Diagnosis Ontology")))
g_tbox.add((ONTO, OWL.versionInfo, Literal("1.0.0")))

# (URI, label, description)
CLASSES = [
    (WD.Product, "Product", "Manufacturable product family / catalog product."),
    (WD.Model, "Model", "Engineering model under a product family."),
    (WD.SKU, "SKU", "Sellable stock-keeping unit / revision."),
    (WD.Asset, "Asset", "Installed unit identified by serial number."),
    (WD.Symptom, "Symptom", "Customer-observable or technician-observed symptom."),
    (WD.ErrorCode, "ErrorCode", "Device-reported diagnostic error code."),
    (WD.FailureMode, "FailureMode", "Diagnosed failure mode (FMEA-aligned)."),
    (WD.DiagnosticStep, "DiagnosticStep", "Troubleshooting or confirmation step."),
    (WD.Component, "Component", "BOM / subsystem component (product structure)."),
    (WD.Part, "Part", "Serviceable replacement part."),
    (WD.Claim, "Claim", "Warranty claim with confirmed failure."),
    (WD.WarrantyPolicy, "WarrantyPolicy", "Coverage rules for parts and labor."),
]

for uri, label, comment in CLASSES:
    g_tbox.add((uri, RDF.type, OWL.Class))
    g_tbox.add((uri, RDFS.label, Literal(label)))
    g_tbox.add((uri, RDFS.comment, Literal(comment)))

print(f"Classes defined: {len(CLASSES)}")
print(f"TBox triples so far: {len(g_tbox)}")

Classes defined: 12
TBox triples so far: 39


In [30]:
# ── TBox Part B: ObjectProperties (edges between nodes) ───────────────────────
#
# ObjectProperty  = both sides are URIs (node → node)
# DatatypeProperty = right side is a literal (node → string/number/date)
#
# rdfs:domain = the "from" class    (left side of the arrow)
# rdfs:range  = the "to" class      (right side of the arrow)
#
# Turtle example:
#   wd:hasSymptom a owl:ObjectProperty ;
#     rdfs:domain wd:Product ;   ← a Product ...
#     rdfs:range  wd:Symptom .   ← ... hasSymptom → a Symptom

OBJECT_PROPS = [
    # (property_URI,          from_class,        to_class,           comment)
    (WD.hasModel, WD.Product, WD.Model, "Product → Model"),
    (WD.hasSku, WD.Product, WD.SKU, "Product → SKU"),
    (WD.modelHasSku, WD.Model, WD.SKU, "Model → SKU revision"),
    (WD.instanceOf, WD.Asset, WD.Product, "Asset is instance of Product"),
    (WD.boundToSku, WD.Asset, WD.SKU, "Asset bound to SKU"),
    (WD.hasSymptom, WD.Product, WD.Symptom, "Product can present Symptom"),
    (WD.hasErrorCode, WD.Product, WD.ErrorCode, "Product can raise ErrorCode"),
    (WD.canHave, WD.Product, WD.FailureMode, "Product can exhibit FailureMode"),
    (WD.hasDiagnosticStep, WD.Product, WD.DiagnosticStep, "Product has DiagnosticStep"),
    (WD.indicates, WD.Symptom, WD.FailureMode, "Symptom → FailureMode (confidence on edge)"),
    (WD.errorCodeIndicates, WD.ErrorCode, WD.FailureMode, "ErrorCode → FailureMode"),
    (WD.confirms, WD.DiagnosticStep, WD.FailureMode, "Step confirms FailureMode"),
    (WD.rulesOut, WD.DiagnosticStep, WD.FailureMode, "Step rules out FailureMode"),
    (WD.nextStep, WD.DiagnosticStep, WD.DiagnosticStep, "Step chains to next Step"),
    (WD.impactsComponent, WD.FailureMode, WD.Component, "FailureMode → Component"),
    (WD.realizedBy, WD.Component, WD.Part, "Component → service Part"),
    (WD.requiresPart, WD.FailureMode, WD.Part, "FailureMode requires Part"),
    (WD.compatibleWith, WD.SKU, WD.Part, "SKU compatible with Part"),
    (WD.coveredBy, WD.Asset, WD.WarrantyPolicy, "Asset covered by WarrantyPolicy"),
]

for prop, domain, range_, comment in OBJECT_PROPS:
    g_tbox.add((prop, RDF.type, OWL.ObjectProperty))
    g_tbox.add((prop, RDFS.domain, domain))
    g_tbox.add((prop, RDFS.range, range_))
    g_tbox.add((prop, RDFS.comment, Literal(comment)))

print(f"ObjectProperties added: {len(OBJECT_PROPS)}")

ObjectProperties added: 19


In [31]:
# ── TBox Part C: DatatypeProperties (attributes with literal values) ───────────
#
# DatatypeProperty = right side is NOT another node, it is a literal value
# rdfs:range here is an XSD datatype, not an owl:Class
#
# XSD type cheat sheet:
#   xsd:string    → any text
#   xsd:decimal   → number with decimals (0.94, 28.99)
#   xsd:integer   → whole number (24, 1)
#   xsd:boolean   → true / false
#   xsd:date      → ISO date (2024-01-15)
#
# Turtle example:
#   wd:severity a owl:DatatypeProperty ;
#     rdfs:domain wd:Symptom ;
#     rdfs:range  xsd:string .   ← value will be a string, not a URI

DATATYPE_PROPS = [
    # (property_URI,         domain,            xsd_type,     comment)
    (WD.productId, WD.Product, XSD.string, "Stable product identifier"),
    (WD.name, WD.Product, XSD.string, "Display name"),
    (WD.category, WD.Product, XSD.string, "Product category"),
    (WD.brand, WD.Product, XSD.string, "Brand name"),
    (WD.symptomId, WD.Symptom, XSD.string, "Symptom identifier"),
    (WD.description, WD.Symptom, XSD.string, "Symptom description text"),
    (WD.severity, WD.Symptom, XSD.string, "low | medium | high | critical"),
    (WD.errorCodeId, WD.ErrorCode, XSD.string, "Error code string e.g. E21"),
    (WD.failureModeId, WD.FailureMode, XSD.string, "Failure mode identifier"),
    (WD.confidence, None, XSD.decimal, "Link confidence 0.0-1.0 (used on Axiom nodes)"),
    (WD.partNumber, WD.Part, XSD.string, "OEM / service part number"),
    (WD.estimatedCostUsd, WD.Part, XSD.decimal, "Estimated part cost in USD"),
    (WD.stepId, WD.DiagnosticStep, XSD.string, "Step identifier"),
    (WD.instruction, WD.DiagnosticStep, XSD.string, "Step instruction text"),
    (WD.serialNumber, WD.Asset, XSD.string, "Asset serial number"),
    (WD.policyId, WD.WarrantyPolicy, XSD.string, "Policy identifier"),
    (WD.coverageMonths, WD.WarrantyPolicy, XSD.integer, "Coverage duration in months"),
]

for prop, domain, range_, comment in DATATYPE_PROPS:
    g_tbox.add((prop, RDF.type, OWL.DatatypeProperty))
    if domain:  # confidence has no fixed domain
        g_tbox.add((prop, RDFS.domain, domain))
    g_tbox.add((prop, RDFS.range, range_))
    g_tbox.add((prop, RDFS.comment, Literal(comment)))

print(f"DatatypeProperties added: {len(DATATYPE_PROPS)}")
print(f"Total TBox triples now:   {len(g_tbox)}")

DatatypeProperties added: 17
Total TBox triples now:   182


### OWL-Only Axioms

Plain RDF records facts. OWL adds **logical rules** a reasoner can act on.  
You don't need to run a reasoner for this project — these axioms document intent.

| OWL construct | What it tells a reasoner | Written as |
|---|---|---|
| `owl:disjointWith` | Two classes can never share a member | `Product owl:disjointWith Symptom` |
| `owl:inverseOf` | A→B automatically implies B←A | `isModelOf owl:inverseOf hasModel` |
| `owl:TransitiveProperty` | A→B, B→C implies A→C | `nextStep a owl:TransitiveProperty` |
| `owl:Restriction` | Instances must satisfy a cardinality rule | Asset must have ≥1 WarrantyPolicy |
| `owl:Axiom` (blank node) | Attach metadata to a triple | `confidence` on an `indicates` edge |

In [32]:
# ── TBox Part D: OWL-only logical axioms ──────────────────────────────────────

# 1. Disjoint classes — a reasoner flags anything that is BOTH a Product AND a Symptom
for class_a, class_b in [
    (WD.Product, WD.Symptom),
    (WD.Product, WD.FailureMode),
    (WD.Symptom, WD.FailureMode),
    (WD.Part, WD.Component),
]:
    g_tbox.add((class_a, OWL.disjointWith, class_b))

# 2. Inverse property
#    If product_wm001 hasModel model_wm8k  →  reasoner infers  model_wm8k isModelOf product_wm001
g_tbox.add((WD.isModelOf, RDF.type, OWL.ObjectProperty))
g_tbox.add((WD.isModelOf, OWL.inverseOf, WD.hasModel))
g_tbox.add((WD.isModelOf, RDFS.label, Literal("isModelOf")))

# 3. Transitive property
#    stepA nextStep stepB  +  stepB nextStep stepC  →  stepA nextStep stepC  (inferred)
g_tbox.add((WD.nextStep, RDF.type, OWL.TransitiveProperty))

# 4. Cardinality restriction using a blank node (anonymous node, no name needed)
#    "Every Asset MUST have at least one WarrantyPolicy"
#    In Turtle:  wd:Asset rdfs:subClassOf [ a owl:Restriction ;
#                  owl:onProperty wd:coveredBy ; owl:minCardinality 1 ] .
r = BNode()  # anonymous blank node
g_tbox.add((r, RDF.type, OWL.Restriction))
g_tbox.add((r, OWL.onProperty, WD.coveredBy))
g_tbox.add((r, OWL.minCardinality, Literal(1, datatype=XSD.integer)))
g_tbox.add((WD.Asset, RDFS.subClassOf, r))  # Asset must satisfy this restriction

print("OWL axioms added ✓")
print(f"Final TBox triple count: {len(g_tbox)}")

OWL axioms added ✓
Final TBox triple count: 194


In [33]:
# ── Serialize TBox to Turtle — READ this to see what you just built ────────────
# Every add() call above corresponds to one or more of these lines.
print(g_tbox.serialize(format="turtle"))

@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix wd: <https://example.org/warranty-diagnosis#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

wd:Claim a owl:Class ;
    rdfs:label "Claim" ;
    rdfs:comment "Warranty claim with confirmed failure." .

wd:boundToSku a owl:ObjectProperty ;
    rdfs:comment "Asset bound to SKU" ;
    rdfs:domain wd:Asset ;
    rdfs:range wd:SKU .

wd:brand a owl:DatatypeProperty ;
    rdfs:comment "Brand name" ;
    rdfs:domain wd:Product ;
    rdfs:range xsd:string .

wd:canHave a owl:ObjectProperty ;
    rdfs:comment "Product can exhibit FailureMode" ;
    rdfs:domain wd:Product ;
    rdfs:range wd:FailureMode .

wd:category a owl:DatatypeProperty ;
    rdfs:comment "Product category" ;
    rdfs:domain wd:Product ;
    rdfs:range xsd:string .

wd:compatibleWith a owl:ObjectProperty ;
    rdfs:comment "SKU compatible with Part" ;
    rdfs:domain wd:SKU ;
    rdfs:range wd:Part .

wd:con

---
## Part 2 — ABox: Populating the Graph with Data

The TBox defined the vocabulary. The ABox uses that vocabulary to record **actual instances**.

```turtle
# TBox line (defines the class Product):
wd:Product  a  owl:Class .                           

# ABox lines (create an instance of Product):
wd:product_wm001  a            wd:Product .          # ← "this node IS a Product"
wd:product_wm001  wd:name      "Washing Machine" .   # ← data attribute
wd:product_wm001  wd:hasModel  wd:model_wm8k .       # ← relationship to another node
```

### Annotating edges: `owl:Axiom` reification

In a graph you can add properties to *nodes* easily. To add metadata to an *edge* (like `confidence: 0.94` on the `indicates` relationship), you use a **blank node** that "points at" the triple:

```turtle
# The plain edge:
wd:sym_wm_s01  wd:indicates  wd:fm_wm_fm01 .

# The annotation ([] = anonymous blank node):
[]  a  owl:Axiom ;
    owl:annotatedSource    wd:sym_wm_s01 ;     ← points at the subject
    owl:annotatedProperty  wd:indicates ;       ← points at the predicate
    owl:annotatedTarget    wd:fm_wm_fm01 ;      ← points at the object
    wd:confidence          "0.94"^^xsd:decimal .  ← the extra data
```

In Neo4j this maps to a **relationship property**: `(s)-[:INDICATES {confidence: 0.94}]->(o)`

In [34]:
# ── ABox: Products, Models, SKUs, Assets ──────────────────────────────────────
g_abox = new_graph()

# ── Product ───────────────────────────────────────────────────────────────────
g_abox.add((WD.product_wm001, RDF.type, WD.Product))
g_abox.add((WD.product_wm001, WD.productId, Literal("wm-001")))
g_abox.add((WD.product_wm001, WD.name, Literal("Front Load Washing Machine 8kg")))
g_abox.add((WD.product_wm001, WD.category, Literal("Laundry")))
g_abox.add((WD.product_wm001, WD.brand, Literal("AquaHome")))

# ── Model ─────────────────────────────────────────────────────────────────────
g_abox.add((WD.model_wm8k, RDF.type, WD.Model))
g_abox.add((WD.model_wm8k, RDFS.label, Literal("AH-WM8K")))
g_abox.add((WD.product_wm001, WD.hasModel, WD.model_wm8k))  # relationship triple

# ── SKUs (two revisions of the same model) ────────────────────────────────────
for sku_attr, sku_label in [("sku_wm8k_2023", "SKU-WM8K-2023"), ("sku_wm8k_2024", "SKU-WM8K-2024")]:
    sku_uri = WD[sku_attr]
    g_abox.add((sku_uri, RDF.type, WD.SKU))
    g_abox.add((sku_uri, RDFS.label, Literal(sku_label)))
    g_abox.add((WD.product_wm001, WD.hasSku, sku_uri))
    g_abox.add((WD.model_wm8k, WD.modelHasSku, sku_uri))

# ── Asset (a specific installed unit with a serial number) ────────────────────
g_abox.add((WD.asset_SN00123, RDF.type, WD.Asset))
g_abox.add((WD.asset_SN00123, WD.serialNumber, Literal("SN-00123")))
g_abox.add((WD.asset_SN00123, WD.instanceOf, WD.product_wm001))
g_abox.add((WD.asset_SN00123, WD.boundToSku, WD["sku_wm8k_2024"]))

# ── WarrantyPolicy ────────────────────────────────────────────────────────────
g_abox.add((WD.policy_WP001, RDF.type, WD.WarrantyPolicy))
g_abox.add((WD.policy_WP001, WD.policyId, Literal("WP-001")))
g_abox.add((WD.policy_WP001, WD.coverageMonths, Literal(24, datatype=XSD.integer)))
g_abox.add((WD.asset_SN00123, WD.coveredBy, WD.policy_WP001))

print(f"Product/Model/SKU/Asset triples: {len(g_abox)}")

Product/Model/SKU/Asset triples: 24


In [35]:
# ── ABox: Symptoms + FailureModes ─────────────────────────────────────────────

SYMPTOMS = [
    ("sym_wm_s01", "wm-s01", "Machine does not spin during final cycle", "high"),
    ("sym_wm_s02", "wm-s02", "Excessive vibration and banging noise", "medium"),
    ("sym_wm_s03", "wm-s03", "Water remains in drum after cycle", "high"),
    ("sym_wm_s04", "wm-s04", "Error code E21 displayed on panel", "medium"),
]
for attr, sym_id, desc, sev in SYMPTOMS:
    uri = WD[attr]
    g_abox.add((uri, RDF.type, WD.Symptom))
    g_abox.add((uri, WD.symptomId, Literal(sym_id)))
    g_abox.add((uri, WD.description, Literal(desc)))
    g_abox.add((uri, WD.severity, Literal(sev)))
    g_abox.add((WD.product_wm001, WD.hasSymptom, uri))

FAILURE_MODES = [
    ("fm_wm_fm01", "wm-fm01", "Worn Drive Belt", "Drive belt stretched or snapped, preventing drum rotation."),
    (
        "fm_wm_fm02",
        "wm-fm02",
        "Failed Drain Pump",
        "Pump impeller blocked or motor burned out, preventing water evacuation.",
    ),
    (
        "fm_wm_fm03",
        "wm-fm03",
        "Unbalanced Load Sensor Fault",
        "Suspension rods or shock absorbers degraded, triggering imbalance shutdown.",
    ),
]
for attr, fm_id, label, comment in FAILURE_MODES:
    uri = WD[attr]
    g_abox.add((uri, RDF.type, WD.FailureMode))
    g_abox.add((uri, WD.failureModeId, Literal(fm_id)))
    g_abox.add((uri, RDFS.label, Literal(label)))
    g_abox.add((uri, RDFS.comment, Literal(comment)))
    g_abox.add((WD.product_wm001, WD.canHave, uri))

# Plain symptom → failureMode edges (without confidence yet)
g_abox.add((WD.sym_wm_s01, WD.indicates, WD.fm_wm_fm01))  # no-spin    → worn belt
g_abox.add((WD.sym_wm_s02, WD.indicates, WD.fm_wm_fm03))  # vibration  → sensor fault
g_abox.add((WD.sym_wm_s03, WD.indicates, WD.fm_wm_fm02))  # water left → drain pump
g_abox.add((WD.sym_wm_s04, WD.indicates, WD.fm_wm_fm01))  # E21 code   → worn belt

print(f"ABox after symptoms + failure modes: {len(g_abox)} triples")

ABox after symptoms + failure modes: 63 triples


In [36]:
# ── ABox: Reification — add confidence scores to edges via owl:Axiom ──────────
#
# Pattern:
#   1. The plain triple already exists: (sym, WD.indicates, fm)
#   2. Create an anonymous BNode (no URI needed)
#   3. Add 4 triples to the BNode: type=Axiom, source=sym, property=indicates, target=fm
#   4. Add the metadata: confidence value

CONFIDENCE_EDGES = [
    (WD.sym_wm_s01, WD.indicates, WD.fm_wm_fm01, "0.94"),
    (WD.sym_wm_s02, WD.indicates, WD.fm_wm_fm03, "0.85"),
    (WD.sym_wm_s03, WD.indicates, WD.fm_wm_fm02, "0.90"),
    (WD.sym_wm_s04, WD.indicates, WD.fm_wm_fm01, "0.55"),
]

for source, prop, target, conf in CONFIDENCE_EDGES:
    axiom = BNode()  # anonymous node — no name
    g_abox.add((axiom, RDF.type, OWL.Axiom))
    g_abox.add((axiom, OWL.annotatedSource, source))  # points at subject
    g_abox.add((axiom, OWL.annotatedProperty, prop))  # points at predicate
    g_abox.add((axiom, OWL.annotatedTarget, target))  # points at object
    g_abox.add((axiom, WD.confidence, Literal(conf, datatype=XSD.decimal)))

print("Confidence annotations added ✓")
print(f"ABox triples now: {len(g_abox)}")

Confidence annotations added ✓
ABox triples now: 83


In [37]:
# ── ABox: Parts + DiagnosticSteps ─────────────────────────────────────────────

PARTS = [
    ("part_belt_wm8k", "AH-WM8K-BELT-001", "Drive Belt WM8K", "28.99", WD.fm_wm_fm01),
    ("part_pump_wm8k", "AH-WM8K-PUMP-002", "Drain Pump WM8K", "65.00", WD.fm_wm_fm02),
    ("part_rod_wm8k", "AH-WM8K-ROD-003", "Suspension Rod WM8K", "18.50", WD.fm_wm_fm03),
]
for attr, pn, label, cost, fm in PARTS:
    uri = WD[attr]
    g_abox.add((uri, RDF.type, WD.Part))
    g_abox.add((uri, WD.partNumber, Literal(pn)))
    g_abox.add((uri, RDFS.label, Literal(label)))
    g_abox.add((uri, WD.estimatedCostUsd, Literal(cost, datatype=XSD.decimal)))
    g_abox.add((fm, WD.requiresPart, uri))

# DiagnosticSteps chained together with nextStep
STEPS = [
    ("step_wm_ds01", "wm-ds01", "Open back panel, inspect the drive belt for cracks or slack.", WD.fm_wm_fm01),
    ("step_wm_ds02", "wm-ds02", "Run drain-only cycle and listen for pump motor hum or grinding.", WD.fm_wm_fm02),
    ("step_wm_ds03", "wm-ds03", "Check suspension rods for wear; push drum and observe rebound.", WD.fm_wm_fm03),
]
prev = None
for attr, sid, inst, confirms_fm in STEPS:
    uri = WD[attr]
    g_abox.add((uri, RDF.type, WD.DiagnosticStep))
    g_abox.add((uri, WD.stepId, Literal(sid)))
    g_abox.add((uri, WD.instruction, Literal(inst)))
    g_abox.add((uri, WD.confirms, confirms_fm))
    g_abox.add((WD.product_wm001, WD.hasDiagnosticStep, uri))
    if prev:
        g_abox.add((prev, WD.nextStep, uri))  # chain: ds01 → ds02 → ds03
    prev = uri

print(f"Final ABox triple count: {len(g_abox)}")

Final ABox triple count: 115


In [38]:
# ── Merge TBox + ABox and serialize to files ──────────────────────────────────
g_full = g_tbox + g_abox

os.makedirs("generated_ontology", exist_ok=True)

g_tbox.serialize(destination="generated_ontology/warranty-schema.ttl", format="turtle")
g_abox.serialize(destination="generated_ontology/warranty-abox.ttl", format="turtle")
g_full.serialize(destination="generated_ontology/warranty-full.ttl", format="turtle")
g_full.serialize(destination="generated_ontology/warranty.owl", format="xml")  # same data, XML syntax

print("Files written:")
for fn in sorted(os.listdir("generated_ontology")):
    size = os.path.getsize(f"generated_ontology/{fn}")
    print(f"  {fn:40s}  {size:>7,} bytes")
print(f"\nTotal triples (TBox + ABox): {len(g_full)}")

Files written:
  warranty-abox.ttl                           4,349 bytes
  warranty-full.ttl                          11,440 bytes
  warranty-schema.ttl                         7,303 bytes
  warranty.owl                               30,228 bytes

Total triples (TBox + ABox): 309


---
## Part 3 — SPARQL: Querying the Graph

SPARQL is to RDF what SQL is to relational databases.

| SQL | SPARQL equivalent |
|-----|------------------|
| `SELECT col1, col2` | `SELECT ?col1 ?col2` (variables start with `?`) |
| `FROM table` | `WHERE { ?s rdf:type wd:Product }` |
| `WHERE a = b` | `?s wd:name ?name` (triple pattern) |
| `JOIN` | Just write more triple patterns — shared variables join them |
| `ORDER BY col DESC` | `ORDER BY DESC(?conf)` |
| `LIMIT 10` | `LIMIT 10` |
| `LEFT JOIN` | `OPTIONAL { ... }` |

### Minimal template
```sparql
PREFIX wd: <https://example.org/warranty-diagnosis#>

SELECT ?a ?b
WHERE {
  ?subject  wd:someProperty  ?a .   ← triple pattern (. ends it)
  ?subject  rdfs:label       ?b .   ← same ?subject = implicit JOIN
}
```

In [39]:
# ── Reusable prefix block ──────────────────────────────────────────────────────
PFX = """
PREFIX wd:   <https://example.org/warranty-diagnosis#>
PREFIX rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
"""

# ── Query 1: All symptoms of the washing machine ──────────────────────────────
# Reading: "Find ?sym where product_wm001 hasSymptom ?sym, and get its id/desc/sev"
q1 = (
    PFX
    + """
SELECT ?id ?description ?severity
WHERE {
  wd:product_wm001  wd:hasSymptom  ?sym .
  ?sym  wd:symptomId   ?id .
  ?sym  wd:description ?description .
  ?sym  wd:severity    ?severity .
}
ORDER BY DESC(?severity)
"""
)

print("=== Q1: Symptoms for product_wm001 ===")
for row in g_full.query(q1):
    print(f"  [{row.severity:6s}]  {row.id}:  {row.description}")

=== Q1: Symptoms for product_wm001 ===
  [medium]  wm-s04:  Error code E21 displayed on panel
  [medium]  wm-s02:  Excessive vibration and banging noise
  [high  ]  wm-s01:  Machine does not spin during final cycle
  [high  ]  wm-s03:  Water remains in drum after cycle


In [40]:
# ── Query 2: Symptom → FailureMode with confidence from the reified Axiom ──────
# The owl:Axiom blank node lets us access edge metadata (confidence) in a query.
q2 = (
    PFX
    + """
SELECT ?symDesc ?fmLabel ?conf
WHERE {
  ?sym  wd:indicates   ?fm .
  ?sym  wd:description ?symDesc .
  ?fm   rdfs:label     ?fmLabel .

  ?axiom  a  owl:Axiom ;
    owl:annotatedSource    ?sym ;
    owl:annotatedProperty  wd:indicates ;
    owl:annotatedTarget    ?fm ;
    wd:confidence          ?conf .
}
ORDER BY DESC(?conf)
"""
)

print("=== Q2: Symptom → FailureMode ranked by confidence ===")
for row in g_full.query(q2):
    conf_val = float(str(row.conf))
    bar = "█" * int(conf_val * 10)
    print(f"  {conf_val:.2f}  {bar:10s}  {str(row.fmLabel):32s}  ←  {str(row.symDesc)[:42]}")

=== Q2: Symptom → FailureMode ranked by confidence ===
  0.94  █████████   Worn Drive Belt                   ←  Machine does not spin during final cycle
  0.90  █████████   Failed Drain Pump                 ←  Water remains in drum after cycle
  0.85  ████████    Unbalanced Load Sensor Fault      ←  Excessive vibration and banging noise
  0.55  █████       Worn Drive Belt                   ←  Error code E21 displayed on panel


In [41]:
# ── Query 3: Full diagnostic path — symptom → FM → step → part + cost ─────────
# Multiple triple patterns with a shared variable chain = implicit JOINs
q3 = (
    PFX
    + """
SELECT ?symptomDesc ?fmLabel ?instruction ?partLabel ?cost
WHERE {
  wd:product_wm001  wd:hasSymptom  ?sym .
  ?sym  wd:description  ?symptomDesc .
  ?sym  wd:indicates    ?fm .
  ?fm   rdfs:label      ?fmLabel .
  ?fm   wd:requiresPart ?part .
  ?part rdfs:label      ?partLabel .
  ?part wd:estimatedCostUsd ?cost .
  OPTIONAL {
    ?step  wd:confirms    ?fm .
    ?step  wd:instruction ?instruction .
  }
}
ORDER BY ?fmLabel
"""
)

print("=== Q3: Full diagnostic paths ===")
seen = set()
for row in g_full.query(q3):
    key = str(row.fmLabel)
    if key not in seen:
        seen.add(key)
        print(f"\n  Failure Mode : {row.fmLabel}")
        print(f"  Symptom      : {str(row.symptomDesc)[:55]}")
        print(f"  Part needed  : {row.partLabel}  (${row.cost})")
        if row.instruction:
            print(f"  Diag step    : {str(row.instruction)[:60]}")

=== Q3: Full diagnostic paths ===

  Failure Mode : Failed Drain Pump
  Symptom      : Water remains in drum after cycle
  Part needed  : Drain Pump WM8K  ($65.00)
  Diag step    : Run drain-only cycle and listen for pump motor hum or grindi

  Failure Mode : Unbalanced Load Sensor Fault
  Symptom      : Excessive vibration and banging noise
  Part needed  : Suspension Rod WM8K  ($18.50)
  Diag step    : Check suspension rods for wear; push drum and observe reboun

  Failure Mode : Worn Drive Belt
  Symptom      : Error code E21 displayed on panel
  Part needed  : Drive Belt WM8K  ($28.99)
  Diag step    : Open back panel, inspect the drive belt for cracks or slack.


---
## Part 4 — LangGraph Diagnostic Agent

### Architecture
```
Human message
      │
      ▼
 ┌──────────────────────┐
 │  diagnostic_agent    │  ← Beat 4: LLM decides WHETHER to call a tool
 └──────────┬───────────┘
            │
    ┌───────┴────────┐
    │                │
 tool_calls?      no calls
    │                │
    ▼                ▼
 ┌───────┐         END
 │ tools │  ← Beat 4: ToolNode runs query_diagnostic_graph
 └───┬───┘
     └──────────────► back to diagnostic_agent
```

### The 5 Beats
| Beat | Job | This notebook |
|------|-----|---------------|
| **1** | Backend connection | `g_full` — our rdflib graph |
| **2** | Tool definition | `@tool query_diagnostic_graph` generates SPARQL |
| **3** | State definition | `AgentState` — the message bag |
| **4** | Nodes + graph wiring | `StateGraph` with conditional edge |
| **5** | Run | `app.invoke(state)` |

### Two modes
- **Mock mode** (no API key): keyword router generates SPARQL — full LangGraph still runs
- **Real LLM mode** (set `OPENAI_API_KEY`): GPT-4o-mini generates SPARQL

In [42]:
# ── BEAT 1: Imports + mode detection ─────────────────────────────────────────
import operator
import os
from typing import Annotated, TypedDict

from langchain_core.messages import AIMessage, ToolCall
from langchain_core.tools import tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, StateGraph
from langgraph.prebuilt import ToolNode
from pydantic import BaseModel, Field

USE_REAL_LLM = bool(os.getenv("OPENAI_API_KEY"))

if USE_REAL_LLM:
    from langchain_openai import ChatOpenAI

    print("✓ OPENAI_API_KEY found — real LLM mode")
else:
    print("⚠ No OPENAI_API_KEY — mock mode (full LangGraph still runs)")

⚠ No OPENAI_API_KEY — mock mode (full LangGraph still runs)


In [43]:
# ── BEAT 2a: Mock SPARQL generator ───────────────────────────────────────────
# In real LLM mode, ChatOpenAI generates these queries.
# In mock mode, this keyword router does the same job so LangGraph still runs.
# The graph wiring (Beats 3–5) is IDENTICAL either way.


def mock_sparql_generator(user_query: str) -> str:
    """Rule-based SPARQL generator. Maps keywords → queries against g_full."""
    q = user_query.lower()

    if any(
        w in q for w in ["symptom", "problem", "issue", "not spin", "vibrat", "water", "noise", "bang", "error code"]
    ):
        return (
            PFX
            + """
SELECT ?id ?description ?severity WHERE {
  wd:product_wm001 wd:hasSymptom ?sym .
  ?sym wd:symptomId ?id .
  ?sym wd:description ?description .
  ?sym wd:severity ?severity .
} ORDER BY DESC(?severity)
"""
        )
    elif any(w in q for w in ["failure", "cause", "fault", "broken", "wrong", "diagnos", "likely"]):
        return (
            PFX
            + """
SELECT ?fmLabel ?symDesc ?confidence WHERE {
  ?sym wd:indicates ?fm .
  ?fm rdfs:label ?fmLabel .
  ?sym wd:description ?symDesc .
  ?axiom a owl:Axiom ;
    owl:annotatedSource ?sym ;
    owl:annotatedProperty wd:indicates ;
    owl:annotatedTarget ?fm ;
    wd:confidence ?confidence .
} ORDER BY DESC(?confidence)
"""
        )
    elif any(w in q for w in ["part", "replace", "repair", "fix", "cost", "price", "buy", "order", "purchase"]):
        return (
            PFX
            + """
SELECT ?fmLabel ?partLabel ?partNumber ?cost WHERE {
  ?fm rdfs:label ?fmLabel .
  ?fm wd:requiresPart ?part .
  ?part rdfs:label ?partLabel .
  ?part wd:partNumber ?partNumber .
  ?part wd:estimatedCostUsd ?cost .
}
"""
        )
    elif any(w in q for w in ["step", "how to", "check", "inspect", "procedure", "test", "verify", "confirm"]):
        return (
            PFX
            + """
SELECT ?stepId ?instruction ?fmLabel WHERE {
  ?step wd:stepId ?stepId .
  ?step wd:instruction ?instruction .
  ?step wd:confirms ?fm .
  ?fm rdfs:label ?fmLabel .
}
"""
        )
    else:
        return (
            PFX
            + """
SELECT ?symptom ?fmLabel ?partLabel WHERE {
  wd:product_wm001 wd:hasSymptom ?sym .
  ?sym wd:description ?symptom .
  OPTIONAL {
    ?sym wd:indicates ?fm . ?fm rdfs:label ?fmLabel .
    OPTIONAL { ?fm wd:requiresPart ?p . ?p rdfs:label ?partLabel . }
  }
} LIMIT 10
"""
        )


print("Mock SPARQL generator ready ✓")

Mock SPARQL generator ready ✓


In [44]:
# ── BEAT 2b: The @tool — the bridge between natural language and the graph ─────
#
# When the LLM decides it needs graph data, LangGraph calls this function.
# It does two things:
#   Step A — decide WHICH query to run (LLM or mock keyword router)
#   Step B — run it and return results as a plain string
#
# This is a SEPARATE inner LLM call from the outer agent_node LLM (Beat 4).
# Outer LLM: "should I call a tool?"  Inner LLM: "what SPARQL should the tool run?"


class DiagnosticQueryInput(BaseModel):
    user_input: str = Field(..., description="User question about the washing machine")


@tool(args_schema=DiagnosticQueryInput)
def query_diagnostic_graph(user_input: str) -> str:
    """Query the diagnostic knowledge graph. Use for questions about
    symptoms, failure modes, repair parts, costs, or diagnostic steps."""

    # ── Step A: generate the SPARQL query ───────────────────────────────────
    if USE_REAL_LLM:
        sparql_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        schema_hint = """
Graph has: Product, Symptom, FailureMode, DiagnosticStep, Part, WarrantyPolicy nodes.
Key properties:
  wd:product_wm001 wd:hasSymptom ?sym
  ?sym wd:description ?desc . ?sym wd:severity ?sev
  ?sym wd:indicates ?fm . ?fm rdfs:label ?label
  ?fm wd:requiresPart ?part . ?part wd:estimatedCostUsd ?cost
  ?step wd:confirms ?fm . ?step wd:instruction ?inst
For confidence: ?axiom a owl:Axiom ; owl:annotatedSource ?s ;
  owl:annotatedProperty wd:indicates ; owl:annotatedTarget ?o ; wd:confidence ?c .
"""
        prompt = (
            schema_hint + f'\nUser question: "{user_input}"\n'
            "Write a SPARQL SELECT query using these prefixes:\n"
            + PFX
            + "\nReturn ONLY the SPARQL. No markdown. No explanation."
        )
        resp = sparql_llm.invoke(prompt)
        sparql_query = resp.content.strip()
        # Strip markdown code fences if the LLM adds them anyway
        if "```" in sparql_query:
            lines = [l for l in sparql_query.split("\n") if not l.strip().startswith("```")]
            sparql_query = "\n".join(lines)
    else:
        sparql_query = mock_sparql_generator(user_input)

    # ── Step B: run the query ────────────────────────────────────────────────
    try:
        rows = list(g_full.query(sparql_query))
        if not rows:
            return "No results found in the diagnostic graph."
        lines = []
        for row in rows[:15]:  # cap at 15 rows
            lines.append(" | ".join(str(v) for v in row if v is not None))
        return "\n".join(lines)
    except Exception as exc:
        return f"SPARQL error: {exc}\n\nQuery attempted:\n{sparql_query}"


tools = [query_diagnostic_graph]
print("Tool defined ✓  — name:", query_diagnostic_graph.name)

Tool defined ✓  — name: query_diagnostic_graph


In [45]:
# ── BEAT 3: AgentState — the message bag ──────────────────────────────────────
#
# TypedDict defines what state looks like as it flows through the graph.
# Annotated[List, operator.add] means: when two partial states are merged,
# APPEND their message lists (rather than overwrite).
#
# This is the short-term memory: every node sees the FULL conversation history.


class AgentState(TypedDict):
    messages: Annotated[list, operator.add]


# That's it — the entire state is just a growing list of messages.
# Every node receives this dict and returns a dict with a partial update.
print("AgentState defined ✓")

AgentState defined ✓


In [46]:
# ── BEAT 4a: Nodes ────────────────────────────────────────────────────────────


def diagnostic_agent_node(state: AgentState) -> dict:
    """
    NODE 1 — The agent.
    Receives the full message history and either:
      a) Returns a final answer (AIMessage with no tool calls) → graph goes to END
      b) Returns an AIMessage WITH tool_calls → graph routes to the tools node
    """
    if USE_REAL_LLM:
        # Real mode: bind the tool so the LLM knows it CAN call query_diagnostic_graph
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(tools)
        response = llm.invoke(state["messages"])
    else:
        # Mock mode — two-phase logic to avoid infinite recursion:
        #   Phase 1 (first call):  last message is HumanMessage / tuple → invoke the tool
        #   Phase 2 (after tool):  last message is ToolMessage (result arrived) → final answer
        # Without the Phase 2 check the mock always adds tool_calls and loops forever.
        from langchain_core.messages import ToolMessage

        last = state["messages"][-1]

        if isinstance(last, ToolMessage):
            # Tool already ran — wrap the raw graph rows in a readable final answer
            response = AIMessage(content="Based on the diagnostic graph:\n\n" + last.content)
        else:
            # First invocation — extract the user query and call the tool
            content = last[1] if isinstance(last, tuple) else getattr(last, "content", str(last))
            response = AIMessage(
                content="",
                tool_calls=[ToolCall(name="query_diagnostic_graph", args={"user_input": content}, id="mock-tc-001")],
            )
    return {"messages": [response]}


# NODE 2 — ToolNode (provided by langgraph.prebuilt)
# Reads tool_calls from the last AIMessage, calls the functions, returns ToolMessages.
tool_node = ToolNode(tools)


# ── BEAT 4b: Routing function ─────────────────────────────────────────────────
def should_continue(state: AgentState):
    """
    After diagnostic_agent_node runs:
    - If the last message has tool_calls → go to the "tools" node
    - Otherwise → END (agent gave a final answer, loop is done)
    """
    last = state["messages"][-1]
    return "tools" if getattr(last, "tool_calls", None) else END


print("Nodes + routing function defined ✓")

Nodes + routing function defined ✓


In [47]:
# ── BEAT 4c: Wire the StateGraph ──────────────────────────────────────────────
#
# StateGraph is the blueprint. .compile() turns it into a runnable object.
# NOTE: the variable `app_stateless` below has the same structure as the
# reference code's `graph = build_graph()`. Just a different variable name
# to avoid colliding with the rdflib `g_full` graph earlier in this notebook.


def build_workflow(checkpointer=None):
    wf = StateGraph(AgentState)

    # Add nodes
    wf.add_node("diagnostic_agent", diagnostic_agent_node)
    wf.add_node("tools", tool_node)

    # Entry point
    wf.set_entry_point("diagnostic_agent")

    # Conditional edge: after agent runs, go to tools OR end
    wf.add_conditional_edges("diagnostic_agent", should_continue)

    # After tools finish, always loop back to agent
    wf.add_edge("tools", "diagnostic_agent")

    return wf.compile(checkpointer=checkpointer)


# Stateless version (no memory between invocations)
app_stateless = build_workflow()

print("Graph compiled ✓")
print("Nodes:", list(app_stateless.get_graph().nodes.keys()))

Graph compiled ✓
Nodes: ['__start__', 'diagnostic_agent', 'tools', '__end__']


In [48]:
# ── BEAT 5: Run the agent ──────────────────────────────────────────────────────
# Same pattern as the reference code:
#   state = {'messages': [('human', '...')]
#   result = app.invoke(state)
#   print(result['messages'][-1].content)

TEST_QUERIES = [
    "What symptoms does the front load washing machine have?",
    "What are the likely failure modes and how confident are we?",
    "What parts do I need to fix the drain pump, and what do they cost?",
]

for i, query in enumerate(TEST_QUERIES, 1):
    print(f'\n{"="*65}')
    print(f"Test {i}: {query}")
    print("=" * 65)

    initial_state = {"messages": [("human", query)]}
    result = app_stateless.invoke(initial_state)

    # In mock mode the last message is a ToolMessage (raw data)
    # In real LLM mode it is an AIMessage (formatted prose answer)
    final = result["messages"][-1]
    if hasattr(final, "content") and final.content:
        print(final.content[:600])
    else:
        # Find the last message with content
        for msg in reversed(result["messages"]):
            if hasattr(msg, "content") and msg.content:
                print(msg.content[:600])
                break


Test 1: What symptoms does the front load washing machine have?
Based on the diagnostic graph:

wm-s04 | Error code E21 displayed on panel | medium
wm-s02 | Excessive vibration and banging noise | medium
wm-s01 | Machine does not spin during final cycle | high
wm-s03 | Water remains in drum after cycle | high

Test 2: What are the likely failure modes and how confident are we?
Based on the diagnostic graph:

Worn Drive Belt | Machine does not spin during final cycle | 0.94
Failed Drain Pump | Water remains in drum after cycle | 0.90
Unbalanced Load Sensor Fault | Excessive vibration and banging noise | 0.85
Worn Drive Belt | Error code E21 displayed on panel | 0.55

Test 3: What parts do I need to fix the drain pump, and what do they cost?
Based on the diagnostic graph:

Unbalanced Load Sensor Fault | Suspension Rod WM8K | AH-WM8K-ROD-003 | 18.50
Worn Drive Belt | Drive Belt WM8K | AH-WM8K-BELT-001 | 28.99
Failed Drain Pump | Drain Pump WM8K | AH-WM8K-PUMP-002 | 65.00


---
## Part 5 — Memory: Multi-Turn Conversations

### Two kinds of memory in LangGraph

| Memory type | How | Scope |
|-------------|-----|-------|
| **Short-term** | `AgentState.messages` grows each turn | Single `.invoke()` call |
| **Long-term** | `MemorySaver` checkpointer + `thread_id` | Across multiple `.invoke()` calls |

### How `MemorySaver` works

```python
app = workflow.compile(checkpointer=MemorySaver())

# Turn 1
app.invoke({'messages': [('human', 'What symptoms exist?')]},
           config={'configurable': {'thread_id': 'session-01'}})

# Turn 2 — agent REMEMBERS turn 1 because same thread_id
app.invoke({'messages': [('human', 'Which of those is most serious?')]},
           config={'configurable': {'thread_id': 'session-01'}})
```

`thread_id` = a session identifier. One per user conversation.
Different users get different `thread_id`s → completely isolated memory.

In [49]:
# ── Build the graph WITH MemorySaver ──────────────────────────────────────────
# The ONLY difference from build_workflow() above is checkpointer=MemorySaver()

app_with_memory = build_workflow(checkpointer=MemorySaver())
print("Graph with MemorySaver compiled ✓")

Graph with MemorySaver compiled ✓


In [50]:
# ── Multi-turn conversation test ──────────────────────────────────────────────
# Same thread_id = conversation context carries forward.
# "those" / "that" in later turns refers back to earlier answers.

SESSION = {"configurable": {"thread_id": "warranty-session-001"}}

CONVERSATION = [
    "What symptoms does the AquaHome washing machine have?",
    "Which of those symptoms is most likely to indicate a failure mode?",
    "What parts would I need to fix that failure, and how much do they cost?",
]

for turn, msg in enumerate(CONVERSATION, 1):
    print(f"\n--- Turn {turn} ---")
    print(f"Human: {msg}")

    result = app_with_memory.invoke(
        {"messages": [("human", msg)]},
        config=SESSION,  # same thread_id → agent sees full history
    )

    # Print what the agent / tool returned
    final = result["messages"][-1]
    if hasattr(final, "content") and final.content:
        print(f"Agent: {final.content[:400]}")
    else:
        for m in reversed(result["messages"]):
            if hasattr(m, "content") and m.content:
                print(f"Graph: {m.content[:400]}")
                break

total_msgs = len(result["messages"])
print(f"\n--- Conversation history: {total_msgs} messages total ---")


--- Turn 1 ---
Human: What symptoms does the AquaHome washing machine have?
Agent: Based on the diagnostic graph:

wm-s04 | Error code E21 displayed on panel | medium
wm-s02 | Excessive vibration and banging noise | medium
wm-s01 | Machine does not spin during final cycle | high
wm-s03 | Water remains in drum after cycle | high

--- Turn 2 ---
Human: Which of those symptoms is most likely to indicate a failure mode?
Agent: Based on the diagnostic graph:

wm-s04 | Error code E21 displayed on panel | medium
wm-s02 | Excessive vibration and banging noise | medium
wm-s01 | Machine does not spin during final cycle | high
wm-s03 | Water remains in drum after cycle | high

--- Turn 3 ---
Human: What parts would I need to fix that failure, and how much do they cost?
Agent: Based on the diagnostic graph:

Worn Drive Belt | Machine does not spin during final cycle | 0.94
Failed Drain Pump | Water remains in drum after cycle | 0.90
Unbalanced Load Sensor Fault | Excessive vibration and banging n

### What you just built — the full information flow

```
Human: "What parts do I need?"
  └─► AgentState.messages = [HumanMessage("What parts...")]
        └─► diagnostic_agent_node
              └─► LLM sees messages + knows about query_diagnostic_graph tool
              └─► Returns AIMessage(tool_calls=[{name: "query_diagnostic_graph",
                                                 args: {user_input: "What parts..."}]}
                    └─► should_continue → "tools"
                          └─► ToolNode calls query_diagnostic_graph("What parts...")
                                └─► mock_sparql_generator → SPARQL query
                                └─► g_full.query(sparql) → rows
                                └─► Returns ToolMessage("Drive Belt $28.99 | ...")
                                      └─► back to diagnostic_agent_node
                                            └─► LLM sees tool result, writes final answer
                                            └─► Returns AIMessage("You need a Drive Belt...")
                                                  └─► should_continue → END
```

### Concept → code cross-reference

| What you learned | Where it lives in the code |
|------------------|---------------------------|
| Triple = Subject → Predicate → Object | Every `g.add((s, p, o))` |
| TBox = schema | `g_tbox` with `owl:Class`, `owl:ObjectProperty` |
| ABox = instances | `g_abox` with `a wd:Product`, `wd:name Literal(...)` |
| Reification | `BNode()` + `owl:Axiom` pattern |
| LangGraph short-term memory | `AgentState.messages` grows each step |
| LangGraph long-term memory | `MemorySaver()` + `thread_id` in config |
| Tool = the bridge | `@tool query_diagnostic_graph` |
| Conditional routing | `should_continue` + `add_conditional_edges` |

---
## Part 6 — Upgrading to Neo4j (Drop-in Swap)

**The LangGraph graph wiring does NOT change.** Only the backend and tool change.

### RDF → Neo4j mapping

| RDF / Turtle | Neo4j Cypher |
|---|---|
| `wd:product_wm001 a wd:Product` | `CREATE (:Product {id: 'product_wm001'})` |
| `wd:product_wm001 wd:hasSymptom wd:sym_wm_s01` | `MATCH (p),(s) CREATE (p)-[:HAS_SYMPTOM]->(s)` |
| `owl:Axiom` blank node with `wd:confidence 0.94` | `CREATE (s)-[:INDICATES {confidence: 0.94}]->(f)` |
| SPARQL `?s wd:hasSymptom ?sym` | Cypher `(s)-[:HAS_SYMPTOM]->(sym)` |

### To start Neo4j locally
```bash
docker run -p 7474:7474 -p 7687:7687 \
  -e NEO4J_AUTH=neo4j/yourpassword neo4j:latest
```
Then set `USE_NEO4J = True` in the cell below.

In [51]:
import re

# ── Generate Cypher CREATE statements from our ABox ───────────────────────────
# Run these in Neo4j Browser (http://localhost:7474) after starting Neo4j.


def abox_to_cypher(g: Graph) -> list:
    """Convert ABox instance triples to Cypher statements."""
    stmts = []
    seen_nodes = set()

    # 1. CREATE nodes
    for s, _, o in g.triples((None, RDF.type, None)):
        if not isinstance(s, URIRef) or not isinstance(o, URIRef):
            continue
        if str(o).startswith(str(OWL)):  # skip owl:Axiom etc.
            continue
        node_id = str(s).split("#")[-1]
        class_name = str(o).split("#")[-1]
        if node_id in seen_nodes:
            continue
        seen_nodes.add(node_id)

        # Collect literal properties
        props = {"_id": node_id}
        for _, dp, dv in g.triples((s, None, None)):
            if isinstance(dv, Literal):
                props[str(dp).split("#")[-1]] = str(dv)
        prop_str = ", ".join(f"{k}: {repr(v)}" for k, v in props.items())
        stmts.append(f"CREATE (:{class_name} {{{prop_str}}})")

    # 2. CREATE relationships
    skip = {
        RDF.type,
        RDFS.label,
        RDFS.comment,
        RDFS.subClassOf,
        OWL.inverseOf,
        WD.symptomId,
        WD.description,
        WD.severity,
        WD.failureModeId,
        WD.partNumber,
        WD.estimatedCostUsd,
        WD.stepId,
        WD.instruction,
        WD.productId,
        WD.name,
        WD.category,
        WD.brand,
        WD.serialNumber,
        WD.policyId,
        WD.coverageMonths,
        WD.confidence,
        WD.errorCodeId,
    }
    for s, p, o in g.triples((None, None, None)):
        if not (isinstance(s, URIRef) and isinstance(p, URIRef) and isinstance(o, URIRef)):
            continue
        if p in skip:
            continue
        s_id = str(s).split("#")[-1]
        o_id = str(o).split("#")[-1]
        # camelCase property name → UPPER_SNAKE_CASE relationship type
        #   hasSymptom → HAS_SYMPTOM,  requiresPart → REQUIRES_PART, etc.
        prop_name = str(p).split("#")[-1]
        rel = re.sub(r"(?<!^)(?=[A-Z])", "_", prop_name).upper()
        stmts.append(f"MATCH (a {{_id: {repr(s_id)}}}), (b {{_id: {repr(o_id)}}})" f" CREATE (a)-[:{rel}]->(b)")

    return stmts


cypher_stmts = abox_to_cypher(g_abox)
print(f"Generated {len(cypher_stmts)} Cypher statements")
print("\nFirst 12 (paste into Neo4j Browser to load the data):")
for s in cypher_stmts[:12]:
    print(" ", s)

Generated 49 Cypher statements

First 12 (paste into Neo4j Browser to load the data):
  CREATE (:Product {_id: 'product_wm001', productId: 'wm-001', name: 'Front Load Washing Machine 8kg', category: 'Laundry', brand: 'AquaHome'})
  CREATE (:Model {_id: 'model_wm8k', label: 'AH-WM8K'})
  CREATE (:SKU {_id: 'sku_wm8k_2023', label: 'SKU-WM8K-2023'})
  CREATE (:SKU {_id: 'sku_wm8k_2024', label: 'SKU-WM8K-2024'})
  CREATE (:Asset {_id: 'asset_SN00123', serialNumber: 'SN-00123'})
  CREATE (:WarrantyPolicy {_id: 'policy_WP001', policyId: 'WP-001', coverageMonths: '24'})
  CREATE (:Symptom {_id: 'sym_wm_s01', symptomId: 'wm-s01', description: 'Machine does not spin during final cycle', severity: 'high'})
  CREATE (:Symptom {_id: 'sym_wm_s02', symptomId: 'wm-s02', description: 'Excessive vibration and banging noise', severity: 'medium'})
  CREATE (:Symptom {_id: 'sym_wm_s03', symptomId: 'wm-s03', description: 'Water remains in drum after cycle', severity: 'high'})
  CREATE (:Symptom {_id: 'sym_

In [52]:
# ── Load tutorial ABox into Neo4j ─────────────────────────────────────────
# Uses the Cypher statements generated by abox_to_cypher() above,
# then adds confidence scores to INDICATES edges (owl:Axiom data from the ABox).

from neo4j import GraphDatabase

NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

with driver.session() as session:
    # Wipe and reload (safe for this tutorial DB)
    session.run("MATCH (n) DETACH DELETE n")

    # Step 1: CREATE all nodes
    for stmt in cypher_stmts:
        if stmt.startswith("CREATE"):
            session.run(stmt)

    # Step 2: CREATE all relationships
    for stmt in cypher_stmts:
        if stmt.startswith("MATCH"):
            from contextlib import suppress
            with suppress(Exception):
                session.run(stmt)  # skip missing nodes gracefully

    # Step 3: Set confidence on INDICATES edges
    # (abox_to_cypher skips owl:Axiom blank nodes — add the data explicitly)
    CONFIDENCE_MAP = {
        ("sym_wm_s01", "fm_wm_fm01"): 0.94,
        ("sym_wm_s02", "fm_wm_fm03"): 0.85,
        ("sym_wm_s03", "fm_wm_fm02"): 0.90,
        ("sym_wm_s04", "fm_wm_fm01"): 0.55,
    }
    for (sym_id, fm_id), conf in CONFIDENCE_MAP.items():
        session.run(
            "MATCH (s {_id: $s_id})-[r:INDICATES]->(f {_id: $f_id}) " "SET r.confidence = $conf",
            s_id=sym_id,
            f_id=fm_id,
            conf=conf,
        )

    node_count = session.run("MATCH (n) RETURN count(n) AS c").single()["c"]
    rel_count = session.run("MATCH ()-[r]->() RETURN count(r) AS c").single()["c"]

driver.close()
print(f"Neo4j loaded \u2713   nodes={node_count}  relationships={rel_count}")
print("Neo4j Browser: http://localhost:7474  (login: neo4j / password)")
print("Try in Browser: MATCH (s:Symptom)-[r:INDICATES]->(f:FailureMode) RETURN s,r,f")

Neo4j loaded ✓   nodes=19  relationships=30
Neo4j Browser: http://localhost:7474  (login: neo4j / password)
Try in Browser: MATCH (s:Symptom)-[r:INDICATES]->(f:FailureMode) RETURN s,r,f


In [53]:
# ── Neo4j agent — works in mock mode AND with a real LLM ────────────────────
# Identical StateGraph wiring as the rdflib version. Only the @tool body changes.

USE_NEO4J = True
NEO4J_URI = "bolt://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASS = "password"

from langchain_neo4j import Neo4jGraph

neo4j_db = Neo4jGraph(
    url=NEO4J_URI, username=NEO4J_USER, password=NEO4J_PASS, refresh_schema=False
)  # refresh_schema=False → no APOC needed

# ── Mock Cypher generator (same keyword-routing idea as mock_sparql_generator)
# Node properties in Neo4j:
#   Symptom      : _id, description, severity, symptomId
#   FailureMode  : _id, label, failureModeId
#   Part         : _id, label, partNumber, estimatedCostUsd
#   DiagnosticStep: _id, instruction, stepId
# Relationship types: HAS_SYMPTOM, INDICATES (.confidence), REQUIRES_PART,
#   CONFIRMS, NEXT_STEP, HAS_DIAGNOSTIC_STEP, COVERED_BY


def mock_cypher_generator(user_query: str) -> str:
    q = user_query.lower()
    if any(w in q for w in ["symptom", "problem", "issue", "not spin", "vibrat", "water", "noise", "error"]):
        return """MATCH (:Product)-[:HAS_SYMPTOM]->(s:Symptom)
RETURN s.symptomId AS id, s.description AS description, s.severity AS severity
ORDER BY s.severity DESC"""
    elif any(w in q for w in ["failure", "fault", "cause", "broken", "diagnos", "likely", "wrong"]):
        return """MATCH (s:Symptom)-[r:INDICATES]->(fm:FailureMode)
RETURN fm.label AS failureMode, s.description AS symptom, r.confidence AS confidence
ORDER BY r.confidence DESC"""
    elif any(w in q for w in ["part", "replace", "repair", "fix", "cost", "price", "buy", "purchase"]):
        return """MATCH (fm:FailureMode)-[:REQUIRES_PART]->(p:Part)
RETURN fm.label AS failureMode, p.label AS part, p.partNumber AS partNumber,
       p.estimatedCostUsd AS cost"""
    elif any(w in q for w in ["step", "how to", "check", "inspect", "procedure", "test", "verify"]):
        return """MATCH (step:DiagnosticStep)-[:CONFIRMS]->(fm:FailureMode)
RETURN step.stepId AS stepId, step.instruction AS instruction,
       fm.label AS failureMode"""
    else:
        return """MATCH (s:Symptom)-[:INDICATES]->(fm:FailureMode)-[:REQUIRES_PART]->(p:Part)
RETURN s.description AS symptom, fm.label AS failureMode, p.label AS part LIMIT 10"""


# ── @tool backed by Neo4j (same two-step pattern: generate query → run it) ──
class Neo4jQueryInput(BaseModel):
    user_input: str = Field(..., description="User question about the washing machine")


@tool(args_schema=Neo4jQueryInput)
def query_neo4j_graph(user_input: str) -> str:
    """Query the Neo4j diagnostic graph. Use for questions about
    symptoms, failure modes, repair parts, costs, or diagnostic steps."""

    if USE_REAL_LLM:
        cypher_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        prompt = (
            "Neo4j graph: Product, Symptom, FailureMode, DiagnosticStep, Part.\n"
            "Relationships: HAS_SYMPTOM, INDICATES (has .confidence), REQUIRES_PART,\n"
            "  CONFIRMS, NEXT_STEP, HAS_DIAGNOSTIC_STEP.\n"
            "Key properties: Symptom.description, Symptom.severity,\n"
            "  FailureMode.label, Part.label, Part.estimatedCostUsd,\n"
            "  DiagnosticStep.instruction.\n"
            f'User question: "{user_input}"\n'
            "Write ONE Cypher MATCH/RETURN query. No markdown. No explanation."
        )
        resp = cypher_llm.invoke(prompt)
        cypher = resp.content.strip()
    else:
        cypher = mock_cypher_generator(user_input)

    try:
        results = neo4j_db.query(cypher)
        if not results:
            return "No results found."
        lines = []
        for row in results[:15]:
            lines.append(" | ".join(f"{k}: {v}" for k, v in row.items() if v is not None))
        return "\n".join(lines)
    except Exception as exc:
        return f"Cypher error: {exc}\nQuery attempted:\n{cypher}"


# ── Same StateGraph wiring — ONLY the tool and mock phase-check change ────
neo4j_tools = [query_neo4j_graph]
neo4j_tool_node = ToolNode(neo4j_tools)


def neo4j_agent_node(state: AgentState) -> dict:
    if USE_REAL_LLM:
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0).bind_tools(neo4j_tools)
        return {"messages": [llm.invoke(state["messages"])]}
    else:
        from langchain_core.messages import ToolMessage

        last = state["messages"][-1]
        if isinstance(last, ToolMessage):
            return {"messages": [AIMessage(content="Based on Neo4j:\n\n" + last.content)]}
        content = last[1] if isinstance(last, tuple) else getattr(last, "content", str(last))
        return {
            "messages": [
                AIMessage(
                    content="",
                    tool_calls=[ToolCall(name="query_neo4j_graph", args={"user_input": content}, id="neo4j-mock-001")],
                )
            ]
        }


def neo4j_route(state: AgentState):
    return "tools" if getattr(state["messages"][-1], "tool_calls", None) else END


wf = StateGraph(AgentState)
wf.add_node("diagnostic_agent", neo4j_agent_node)
wf.add_node("tools", neo4j_tool_node)
wf.set_entry_point("diagnostic_agent")
wf.add_conditional_edges("diagnostic_agent", neo4j_route)
wf.add_edge("tools", "diagnostic_agent")
app_neo4j = wf.compile(checkpointer=MemorySaver())
print("Neo4j agent compiled \u2713")

# ── Test queries ──────────────────────────────────────────────────────────
NEO4J_THREAD = {"configurable": {"thread_id": "neo4j-session-001"}}

for query in [
    "What symptoms does the washing machine have?",
    "What failure modes are most likely?",
    "What parts do I need and what do they cost?",
]:
    print(f'\n{"="*60}')
    print(f"Q: {query}")
    print("=" * 60)
    result = app_neo4j.invoke({"messages": [("human", query)]}, config=NEO4J_THREAD)
    final = result["messages"][-1]
    if hasattr(final, "content") and final.content:
        print(final.content[:500])
    else:
        for m in reversed(result["messages"]):
            if hasattr(m, "content") and m.content:
                print(m.content[:500])
                break

Neo4j agent compiled ✓

Q: What symptoms does the washing machine have?
Based on Neo4j:

id: wm-s04 | description: Error code E21 displayed on panel | severity: medium
id: wm-s02 | description: Excessive vibration and banging noise | severity: medium
id: wm-s01 | description: Machine does not spin during final cycle | severity: high
id: wm-s03 | description: Water remains in drum after cycle | severity: high

Q: What failure modes are most likely?
Based on Neo4j:

failureMode: Worn Drive Belt | symptom: Machine does not spin during final cycle | confidence: 0.94
failureMode: Failed Drain Pump | symptom: Water remains in drum after cycle | confidence: 0.9
failureMode: Unbalanced Load Sensor Fault | symptom: Excessive vibration and banging noise | confidence: 0.85
failureMode: Worn Drive Belt | symptom: Error code E21 displayed on panel | confidence: 0.55

Q: What parts do I need and what do they cost?
Based on Neo4j:

failureMode: Unbalanced Load Sensor Fault | part: Suspension Rod WM8K